# Peripheral Binding Smoke Tests

Safe smoke-test cells for ASI Tiger and SyncBoard bindings. Run the setup cell first, update the serial port constants, then run one peripheral cell at a time.

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from contextlib import suppress
from pathlib import Path
import logging
import os
import sys
import time
from typing import Any


os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)


def find_workspace_root() -> Path:
    """
    Return the workspace root that contains the evomachine repository.

    Parameters
    ----------
    None

    Returns
    -------
    Path
        Workspace root used to add local sibling repositories to sys.path.
    """
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "evomachine" / "evomachine").is_dir():
            return candidate
    return current


def add_import_path(path: Path) -> None:
    """
    Add an existing package root to sys.path if it is not already present.

    Parameters
    ----------
    path
        Candidate package root to add for notebook imports.

    Returns
    -------
    None
    """
    if path.exists():
        path_text = str(path.resolve())
        if path_text not in sys.path:
            sys.path.insert(0, path_text)


def find_device_by_hwid(hwid: str) -> str:
    """
    Return the serial device whose hardware ID contains the requested value.

    Parameters
    ----------
    hwid
        VID:PID or other identifying text expected in the port hardware ID.

    Returns
    -------
    str
        Device path for the matching serial port.

    Raises
    ------
    ValueError
        If no available serial port contains the requested hardware ID.
    """
    for port in list_serial_ports():
        if hwid in port["hwid"]:
            return port["device"]
    raise ValueError(f"Port with HWID {hwid} not found")



def list_serial_ports() -> list[dict[str, str]]:
    """
    Return available serial ports for choosing hardware constants.

    Parameters
    ----------
    None

    Returns
    -------
    list[dict[str, str]]
        Serial port metadata dictionaries, or one error dictionary if pyserial is unavailable.
    """
    try:
        from serial.tools import list_ports
    except Exception as error:
        return [{"error": f"Could not import pyserial list_ports: {error}"}]
    return [
        {
            "device": port.device,
            "description": port.description,
            "hwid": port.hwid,
        }
        for port in list_ports.comports()
    ]


def require_object(name: str) -> Any:
    """
    Return a global object by name or raise a clear notebook-order error.

    Parameters
    ----------
    name
        Global variable name expected to contain an initialised controller or peripheral.

    Returns
    -------
    Any
        The object stored under name in the notebook global namespace.
    """
    value = globals().get(name)
    if value is None:
        raise RuntimeError(f"Run the {name} setup cell before this cell.")
    return value


def remember_peripheral(name: str, peripheral: Any) -> Any:
    """
    Store a peripheral globally and add it to the cleanup list.

    Parameters
    ----------
    name
        Global variable name that should point to the peripheral.
    peripheral
        Peripheral instance created by a smoke-test cell.

    Returns
    -------
    Any
        The same peripheral instance, for inline assignment.
    """
    globals()[name] = peripheral
    if peripheral not in created_peripherals:
        created_peripherals.append(peripheral)
    return peripheral


def remember_controller(name: str, controller: Any) -> TigerPeripheralController | SyncBoardPeripheralController:
    """
    Store a controller globally and add it to the cleanup list.

    Parameters
    ----------
    name
        Global variable name that should point to the controller.
    controller
        Peripheral controller instance created by a smoke-test cell.

    Returns
    -------
    Any
        The same controller instance, for inline assignment.
    """
    globals()[name] = controller
    if controller not in created_controllers:
        created_controllers.append(controller)
    return controller



WORKSPACE_ROOT = find_workspace_root()
MAIN_REPO_ROOT = WORKSPACE_ROOT / "evomachine"
# add_import_path(MAIN_REPO_ROOT)
# add_import_path(WORKSPACE_ROOT / "asitiger")
# add_import_path(WORKSPACE_ROOT / "sync_board")

from evomachine.bindings.asitiger.autofocus import TigerAutofocus
from evomachine.bindings.asitiger.filterwheel import TigerFilterWheel
from evomachine.bindings.asitiger.leds import TigerLedSource
from evomachine.bindings.asitiger.peripheralcontroller import TigerPeripheralController
from evomachine.bindings.asitiger.stage import TigerStage
from evomachine.bindings.syncboard.leds import SyncBoardLedSource
from evomachine.bindings.syncboard.peripheralcontroller import SyncBoardPeripheralController
from evomachine.bindings.syncboard.photodiode import SyncBoardPhotodiode
from evomachine.coordinates import Coordinate
from evomachine.peripherals.photodiode import PhotodiodeReadingRange
from evomachine.types import FilterWheelType, LEDType


# Route binding logs through each package's single EvoMachine-configured parent logger.
for logger_name in (
    "asitiger.command",
    "asitiger.serialconnection",
    "asitiger.tigercontroller",
    "syncboard.command",
    "syncboard.serialconnection",
    "syncboard.syncboardcontroller",
):
    child_logger = logging.getLogger(logger_name)
    child_logger.handlers.clear()
    child_logger.propagate = True


TIGER_PORT = find_device_by_hwid("10C4:EA60")
SYNCBOARD_PORT = find_device_by_hwid("16C0:0483")

FOV_STEP_SIZE = 100.0
STAGE_SMOKE_COORDINATE = Coordinate(x=0, y=0, z=0)
RUN_STAGE_MOVE = True

TIGER_TEST_LED = LEDType.LED_OVERHEAD_TIGER
SYNCBOARD_TEST_LED = LEDType.LED_385_NM
LED_TEST_BRIGHTNESS = 100
LED_TEST_DURATION_MS = 1000.0

RUN_FILTER_CHANGE = True
TARGET_FILTER = FilterWheelType.FILTER_527nm

RUN_CRISP_CALIBRATION = True
LOCK_AFTER_CRISP_CALIBRATION = False

PHOTODIODE_CHANNEL = 8
PHOTODIODE_READING_RANGE = PhotodiodeReadingRange(0.0, 1.0)

tiger_controller = None
tiger_stage = None
tiger_filter_wheel = None
tiger_led_source = None
tiger_autofocus = None
syncboard_controller = None
syncboard_led_source = None
syncboard_photodiode = None
created_controllers = []
created_peripherals = []

{
    "workspace_root": WORKSPACE_ROOT,
    "main_repo_root": MAIN_REPO_ROOT,
    "serial_ports": list_serial_ports(),
    "tiger_port": TIGER_PORT,
    "syncboard_port": SYNCBOARD_PORT,
}



{'workspace_root': PosixPath('/home/hslab/workspace_python/evomachine_refactor'),
 'main_repo_root': PosixPath('/home/hslab/workspace_python/evomachine_refactor/evomachine'),
 'serial_ports': [{'device': '/dev/ttyS1',
   'description': 'ttyS1',
   'hwid': 'PNP0501'},
  {'device': '/dev/ttyS0', 'description': 'ttyS0', 'hwid': 'PNP0501'},
  {'device': '/dev/ttyUSB0',
   'description': 'CP2102 USB to UART Bridge Controller - CP2102 USB to UART Bridge Controller',
   'hwid': 'USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3'},
  {'device': '/dev/ttyACM0',
   'description': 'USB Serial',
   'hwid': 'USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0'}],
 'tiger_port': '/dev/ttyUSB0',
 'syncboard_port': '/dev/ttyACM0'}

## ASI Tiger

In [2]:
# TigerPeripheralController

tiger_controller = remember_controller(
    "tiger_controller",
    TigerPeripheralController.from_serial_port(port=TIGER_PORT),
)
tiger_controller.initialise()

{
    "name": tiger_controller.name,
    "port": TIGER_PORT,
    "is_initialised": tiger_controller.is_initialised(),
    "is_alive": tiger_controller.is_alive(),
    "card_address_crisp": tiger_controller.card_address_crisp,
    "card_address_led": tiger_controller.card_address_led,
    "card_address_filter_wheel": tiger_controller.card_address_filter_wheel,
}


2026-07-02 09:27:13 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:13 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:13 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:13 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:13 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:13 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'


{'name': 'ASI Tiger Peripheral Controller',
 'port': '/dev/ttyUSB0',
 'is_initialised': True,
 'is_alive': True,
 'card_address_crisp': 2,
 'card_address_led': 7,
 'card_address_filter_wheel': 8}

In [3]:
# TigerStage

tiger_controller = require_object("tiger_controller")
tiger_stage = remember_peripheral(
    "tiger_stage",
    TigerStage(peripheral_ctrl=tiger_controller, fov_step_size=FOV_STEP_SIZE),
)
tiger_stage.initialise()
initial_coordinate = tiger_stage.get_coordinates()
limits = tiger_stage.get_stage_limits()

if RUN_STAGE_MOVE:
    tiger_stage.move(target=STAGE_SMOKE_COORDINATE, block=True)

{
    "name": tiger_stage.name,
    "is_initialised": tiger_stage.is_initialised(),
    "is_alive": tiger_stage.is_alive(),
    "initial_coordinate": initial_coordinate,
    "current_coordinate": tiger_stage.get_coordinates(),
    "stage_limits": limits,
    "ran_stage_move": RUN_STAGE_MOVE,
}


2026-07-02 09:27:17 - DEBUG - evomachine.peripherals.stage - Stage.initialise: initialising ASI Tiger Stage with force=False.
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Sending data: b'W X Y Z\r'
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Received: b':A -152019 56961 61611.7 \r\n'
2026-07-02 09:27:17 - DEBUG - evomachine.peripherals.stage - Stage.initialise: ASI Tiger Stage initialised at (x=-152019.0, y=56961.0, z=61611.7, channel_id=0).
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:17 - DEBUG - asitiger.serialconnection - S

{'name': 'ASI Tiger Stage',
 'is_initialised': True,
 'is_alive': True,
 'initial_coordinate': Coordinate(x=-152019.0, y=56961.0, z=61611.7, channel_id=0),
 'current_coordinate': Coordinate(x=0.0, y=2.0, z=-0.3, channel_id=0),
 'stage_limits': (Coordinate(x=-80000.0, y=-190000.0, z=-10000.0, channel_id=0),
  Coordinate(x=80000.0, y=190000.0, z=10000.0, channel_id=0)),
 'ran_stage_move': True}

In [4]:
# TigerFilterWheel

tiger_controller = require_object("tiger_controller")
available_filters = list(TigerFilterWheel.DEFAULT_FILTER_WHEEL_SETTINGS)
tiger_filter_wheel = remember_peripheral(
    "tiger_filter_wheel",
    TigerFilterWheel(
        peripheral_ctrl=tiger_controller,
        available_filters=available_filters,
    ),
)
tiger_filter_wheel.initialise()
initial_filter = tiger_filter_wheel.get_filter_wheel()

if RUN_FILTER_CHANGE:
    tiger_filter_wheel.set_filter_wheel(filter_type=TARGET_FILTER, force=True)


{
    "name": tiger_filter_wheel.name,
    "is_initialised": tiger_filter_wheel.is_initialised(),
    "is_alive": tiger_filter_wheel.is_alive(),
    "available_filters": tiger_filter_wheel.get_available_filters(),
    "initial_filter": initial_filter,
    "current_filter": tiger_filter_wheel.get_filter_wheel(),
    "ran_filter_change": RUN_FILTER_CHANGE,
}


2026-07-02 09:27:44 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.initialise: initialising ASI Tiger Filter Wheel with force=False.
2026-07-02 09:27:44 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:44 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:44 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:44 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:44 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.initialise: ASI Tiger Filter Wheel initialised at UNKNOWN.
2026-07-02 09:27:44 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:44 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:44 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.set_filter_wheel: setting ASI Tiger Filter Wheel to FILTER_527nm with force=True.
2026-07-02 09:27:44 - DEBUG - asitiger.serialconnection - Sending data: b'8MP 2\r'
2026-

{'name': 'ASI Tiger Filter Wheel',
 'is_initialised': True,
 'is_alive': False,
 'available_filters': [<FilterWheelType.FILTER: 0>,
  <FilterWheelType.FILTER_465nm: 1>,
  <FilterWheelType.FILTER_527nm: 2>,
  <FilterWheelType.FILTER_592nm: 3>,
  <FilterWheelType.NO_FILTER: 4>,
  <FilterWheelType.BLOCKING: 5>],
 'initial_filter': <FilterWheelType.UNKNOWN: -1>,
 'current_filter': <FilterWheelType.FILTER_527nm: 2>,
 'ran_filter_change': True}

In [5]:
# TigerLedSource

tiger_controller = require_object("tiger_controller")
tiger_led_source = remember_peripheral(
    "tiger_led_source",
    TigerLedSource(
        peripheral_ctrl=tiger_controller,
        available_leds=[TIGER_TEST_LED],
    ),
)
tiger_led_source.initialise()
try:
    tiger_led_source.set_led(
        led_type=TIGER_TEST_LED,
        brightness=LED_TEST_BRIGHTNESS,
        duration=LED_TEST_DURATION_MS,
    )
    time.sleep(LED_TEST_DURATION_MS / 1000.0 + 0.05)
finally:
    time.sleep(5)
    tiger_led_source.disable_led()

{
    "name": tiger_led_source.name,
    "is_initialised": tiger_led_source.is_initialised(),
    "is_alive": tiger_led_source.is_alive(),
    "available_leds": tiger_led_source.get_available_leds(),
    "test_led": TIGER_TEST_LED,
    "brightness": LED_TEST_BRIGHTNESS,
    "duration_ms": LED_TEST_DURATION_MS,
    "state_after_disable": tiger_led_source.get_led_state(TIGER_TEST_LED),
}


2026-07-02 09:27:52 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:52 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:52 - DEBUG - evomachine.peripherals.leds - LedSource.initialise: initialising ASI Tiger LED Source with force=False.
2026-07-02 09:27:52 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:52 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:52 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting ASI Tiger LED Source LED_OVERHEAD_TIGER to brightness=100.0 duration=1000.0.
2026-07-02 09:27:52 - DEBUG - asitiger.serialconnection - Sending data: b'7LED Y=100\r'
2026-07-02 09:27:52 - DEBUG - asitiger.serialconnection - Received: b':A\r\n'
2026-07-02 09:27:53 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:27:53 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:27:53 - DEBUG - evomachine.peripherals.leds - LedSourc

{'name': 'ASI Tiger LED Source',
 'is_initialised': True,
 'is_alive': True,
 'available_leds': [<LEDType.LED_OVERHEAD_TIGER: 6>],
 'test_led': <LEDType.LED_OVERHEAD_TIGER: 6>,
 'brightness': 100,
 'duration_ms': 1000.0,
 'state_after_disable': LedState(led_type=<LEDType.LED_OVERHEAD_TIGER: 6>, brightness=0.0, is_on=False, stop_time=None)}

In [6]:
# TigerAutofocus

tiger_controller = require_object("tiger_controller")
tiger_autofocus = remember_peripheral(
    "tiger_autofocus",
    TigerAutofocus(peripheral_ctrl=tiger_controller),
)
tiger_autofocus.initialise()
initial_status = tiger_autofocus.get_status()
calibration_success = None

if RUN_CRISP_CALIBRATION:
    calibration_success = tiger_autofocus.initialise_autofocus(
        lock_after_initialise=LOCK_AFTER_CRISP_CALIBRATION,
    )

{
    "name": tiger_autofocus.name,
    "is_initialised": tiger_autofocus.is_initialised(),
    "is_alive": tiger_autofocus.is_alive(),
    "initial_status": initial_status,
    "current_status": tiger_autofocus.get_status(),
    "is_locked": tiger_autofocus.is_locked(),
    "ran_crisp_calibration": RUN_CRISP_CALIBRATION,
    "calibration_success": calibration_success,
}


2026-07-02 09:28:04 - DEBUG - evomachine.peripherals.autofocus - Autofocus.initialise: initialising ASI Tiger CRISP Autofocus with force=False.
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:28:04 - DEBUG - evomachine.peripherals.autofocus - Autofocus.initialise: ASI Tiger CRISP Autofocus initialised.
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Sending data: b'2LK X?\r'
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Received: b':A I\r\n'
2026-07-02 09:28:04 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 09:28:04

{'name': 'ASI Tiger CRISP Autofocus',
 'is_initialised': True,
 'is_alive': True,
 'initial_status': <AutoFocusStatusType.IDLE: 'I'>,
 'current_status': <AutoFocusStatusType.READY: 'R'>,
 'is_locked': False,
 'ran_crisp_calibration': True,
 'calibration_success': False}

## SyncBoard

In [7]:
# SyncBoardPeripheralController

syncboard_controller = remember_controller(
    "syncboard_controller",
    SyncBoardPeripheralController.from_serial_port(port=SYNCBOARD_PORT),
)
syncboard_controller.initialise()

{
    "name": syncboard_controller.name,
    "port": SYNCBOARD_PORT,
    "is_initialised": syncboard_controller.is_initialised(),
    "is_alive": syncboard_controller.is_alive(),
}


2026-07-02 09:28:34 - DEBUG - syncboard.serialconnection - Connecting to /dev/ttyACM0 at 2000000 baud
2026-07-02 09:28:34 - DEBUG - syncboard.command - Formatted command: $attachLED/true#%
2026-07-02 09:28:34 - DEBUG - syncboard.syncboardcontroller - Sending command $attachLED/true#% at attempt 0.
2026-07-02 09:28:34 - DEBUG - syncboard.serialconnection - Sending data: b'$attachLED/true#%'
2026-07-02 09:28:34 - DEBUG - syncboard.serialconnection - Received: $attachLED/1#%

2026-07-02 09:28:34 - DEBUG - syncboard.serialconnection - Received: ['$attachLED/1#%']
2026-07-02 09:28:34 - INFO - syncboard.serialconnection - SyncBoardController.SerialConnection: syncboard responded with ['$attachLED/1#%'] to $attachLED/true#%.
2026-07-02 09:28:35 - DEBUG - syncboard.serialconnection - Received: 
2026-07-02 09:28:35 - DEBUG - syncboard.command - Formatted command: $attachMagnet/true#%
2026-07-02 09:28:35 - DEBUG - syncboard.syncboardcontroller - Sending command $attachMagnet/true#% at attempt 0.

{'name': 'SyncBoard Peripheral Controller',
 'port': '/dev/ttyACM0',
 'is_initialised': True,
 'is_alive': True}

In [8]:
# SyncBoardLedSource

syncboard_controller = require_object("syncboard_controller")
syncboard_led_source = remember_peripheral(
    "syncboard_led_source",
    SyncBoardLedSource(
        peripheral_ctrl=syncboard_controller,
        available_leds=[SYNCBOARD_TEST_LED],
    ),
)
syncboard_led_source.initialise()
try:
    syncboard_led_source.set_led(
        led_type=SYNCBOARD_TEST_LED,
        brightness=100,
        duration=None,
    )
    time.sleep(LED_TEST_DURATION_MS / 1000.0 + 0.05)
finally:
    time.sleep(5)
    syncboard_led_source.disable_led()

{
    "name": syncboard_led_source.name,
    "is_initialised": syncboard_led_source.is_initialised(),
    "is_alive": syncboard_led_source.is_alive(),
    "available_leds": syncboard_led_source.get_available_leds(),
    "test_led": SYNCBOARD_TEST_LED,
    "brightness": LED_TEST_BRIGHTNESS,
    "duration_ms": LED_TEST_DURATION_MS,
    "state_after_disable": syncboard_led_source.get_led_state(SYNCBOARD_TEST_LED),
}


2026-07-02 09:28:47 - DEBUG - evomachine.peripherals.leds - LedSource.initialise: initialising SyncBoard LED Source with force=False.
2026-07-02 09:28:47 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_385_NM to brightness=100.0 duration=None.
2026-07-02 09:28:47 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is None.
2026-07-02 09:28:47 - DEBUG - syncboard.command - Formatted command: $setupLED/7/0/1.0000000#%
2026-07-02 09:28:47 - DEBUG - syncboard.syncboardcontroller - Sending command $setupLED/7/0/1.0000000#% at attempt 0.
2026-07-02 09:28:47 - DEBUG - syncboard.serialconnection - Sending data: b'$setupLED/7/0/1.0000000#%'
2026-07-02 09:28:47 - DEBUG - syncboard.serialconnection - Received: $Received setup LED command/1#%

2026-07-02 09:28:47 - DEBUG - syncboard.serialconnection - Received: $setupLED/7#%

2026-07-02 09:28:47 - DEBUG - syncboard.serialconnection - Received: ['$Received setup LED

{'name': 'SyncBoard LED Source',
 'is_initialised': True,
 'is_alive': True,
 'available_leds': [<LEDType.LED_385_NM: 0>],
 'test_led': <LEDType.LED_385_NM: 0>,
 'brightness': 100,
 'duration_ms': 1000.0,
 'state_after_disable': LedState(led_type=<LEDType.LED_385_NM: 0>, brightness=0.0, is_on=False, stop_time=None)}

In [33]:
# SyncBoardPhotodiode

syncboard_controller = require_object("syncboard_controller")
syncboard_photodiode = remember_peripheral(
    "syncboard_photodiode",
    SyncBoardPhotodiode(
        peripheral_ctrl=syncboard_controller,
        channel=PHOTODIODE_CHANNEL,
        reading_range=PHOTODIODE_READING_RANGE,
    ),
)
syncboard_photodiode.initialise()
reading = syncboard_photodiode.read_photodiode()

{
    "name": syncboard_photodiode.name,
    "is_initialised": syncboard_photodiode.is_initialised(),
    "is_alive": syncboard_photodiode.is_alive(),
    "channel": syncboard_photodiode.channel,
    "reading_percent": reading,
}


2026-06-30 16:49:26 - DEBUG - evomachine.peripherals.photodiode - Photodiode.initialise: initialising SyncBoard Photodiode with force=False.
2026-06-30 16:49:26 - DEBUG - syncboard.command - Formatted command: $measurePhotodiode/8#%
2026-06-30 16:49:26 - DEBUG - syncboard.command - Formatted command: $measurePhotodiode/8#%
2026-06-30 16:49:26 - DEBUG - syncboard.syncboardcontroller - Sending command $measurePhotodiode/8#% at attempt 0.
2026-06-30 16:49:26 - DEBUG - syncboard.syncboardcontroller - Sending command $measurePhotodiode/8#% at attempt 0.
2026-06-30 16:49:26 - DEBUG - syncboard.serialconnection - Sending data: b'$measurePhotodiode/8#%'
2026-06-30 16:49:26 - DEBUG - syncboard.serialconnection - Sending data: b'$measurePhotodiode/8#%'
2026-06-30 16:49:26 - DEBUG - syncboard.serialconnection - Received: $measurePhotodiode/1.3183475e+00#%

2026-06-30 16:49:26 - DEBUG - syncboard.serialconnection - Received: $measurePhotodiode/1.3183475e+00#%

2026-06-30 16:49:26 - DEBUG - syncboa

RuntimeError: SyncBoardPhotodiode._read_raw_photodiode: received no reading for channel 8.

## Cleanup

In [9]:
# Stop and release any peripherals/controllers created above.

cleanup_errors = []

for peripheral in reversed(created_peripherals):
    with suppress(Exception):
        peripheral.stop()
    try:
        peripheral.finalise()
    except Exception as error:
        cleanup_errors.append((getattr(peripheral, "name", repr(peripheral)), repr(error)))

for controller in reversed(created_controllers):
    try:
        controller.shutdown(force=True)
    except Exception as error:
        cleanup_errors.append((getattr(controller, "name", repr(controller)), repr(error)))

{
    "cleaned_peripherals": [getattr(peripheral, "name", repr(peripheral)) for peripheral in created_peripherals],
    "cleaned_controllers": [getattr(controller, "name", repr(controller)) for controller in created_controllers],
    "cleanup_errors": cleanup_errors,
}


2026-07-02 09:29:11 - DEBUG - evomachine.peripherals.leds - LedSource.stop: disabling all LEDs for SyncBoard LED Source.
2026-07-02 09:29:11 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: disabling LED_385_NM on SyncBoard LED Source.
2026-07-02 09:29:11 - DEBUG - syncboard.command - Formatted command: $switchLED/7/0#%
2026-07-02 09:29:11 - DEBUG - syncboard.syncboardcontroller - Sending command $switchLED/7/0#% at attempt 0.
2026-07-02 09:29:11 - DEBUG - syncboard.serialconnection - Sending data: b'$switchLED/7/0#%'
2026-07-02 09:29:11 - DEBUG - syncboard.serialconnection - Received: Called switchLEDDirect with args channel=7 on=0 force0

2026-07-02 09:29:11 - DEBUG - syncboard.serialconnection - Received: Resetting LED timeout for channel 7

2026-07-02 09:29:11 - DEBUG - syncboard.serialconnection - Received: LED is not timed. Cannot reset timeout.

2026-07-02 09:29:11 - DEBUG - syncboard.serialconnection - Received: $switchLED/7#%

2026-07-02 09:29:11 - DEBUG - syncbo

{'cleaned_peripherals': ['ASI Tiger Stage',
  'ASI Tiger Filter Wheel',
  'ASI Tiger LED Source',
  'ASI Tiger CRISP Autofocus',
  'SyncBoard LED Source'],
 'cleaned_controllers': ['ASI Tiger Peripheral Controller',
  'SyncBoard Peripheral Controller'],
 'cleanup_errors': []}